<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/3_LoRA_%EB%B0%8F_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### GPU 변경 후 실행

In [1]:
#2025.09.15일 현재 최신 버전
!pip install transformers==4.56.1 -qqq
!pip install datasets==4.0.0 -qqq #Dataset 자료형
!pip install huggingface_hub==0.34.4 -qqq
!pip install accelerate==1.10.1 -qqq # 학습 시 자동으로 gpu 최적화 및 gpu 사용 변환등 실행되는 기능
!pip install peft==0.17.1 -qqq #모델을 저비트 양자화 + LoRA 미세조정에 적합하게 바꾸는 역할
!pip install bitsandbytes==0.47.0 -qqq #양자화 - qlora 생성시 필요

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.0 MB/s eta 0:00:00


In [2]:
import transformers
import datasets
import huggingface_hub
import accelerate
import peft
import bitsandbytes
import torch
#import warnings
#warnings.filterwarnings('ignore')
print(f"transformers : {transformers.__version__}" )
print(f"datasets : {datasets.__version__}" )
print(f"huggingface_hub : {huggingface_hub.__version__}" )
print(f"accelerate : {accelerate.__version__}" )
print(f"peft : {peft.__version__}" )
print(f"bitsandbytes : {bitsandbytes.__version__}" )
print(f"torch : {torch.__version__}" )
#경고 : 현재 환경에서는 8-bit 옵티마이저를 사용할 수 없어서 자동으로 일반 옵티마이저(FP32)로 대체된다는 의미
#gpu가 없는 경우 발생 또는 NVIDIA GPU 전용(CUDA) 환경이 아닐 경우 경고 발생

transformers : 4.56.1
datasets : 4.0.0
huggingface_hub : 0.34.4
accelerate : 1.10.1
peft : 0.17.1
bitsandbytes : 0.47.0
torch : 2.10.0+cu128


### LoRA
- 대형 언어 모델(LLM)이나 딥러닝 모델을 효율적으로 미세 조정(Fine-tuning)하는 방법
- 모델 일부에 저차원 행렬만 학습 → 메모리 절약 + 빠른 학습 가능
- 추론 확률을 높이는게 아니라 속도를 빠르게 하기 위한 하나의 방법
- 적용 가능한 층
  - layer층
    - 입력된 데이터에 대해 정답에 가깝게 만드는 연산 단계 층

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "kakaocorp/kanana-nano-2.1b-base"
tokenizer = AutoTokenizer.from_pretrained( model_id )
model = AutoModelForCausalLM.from_pretrained( model_id , device_map="auto" )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [2]:
model.model.layers

ModuleList(
  (0-31): 32 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=1792, out_features=3072, bias=False)
      (k_proj): Linear(in_features=1792, out_features=1024, bias=False)
      (v_proj): Linear(in_features=1792, out_features=1024, bias=False)
      (o_proj): Linear(in_features=3072, out_features=1792, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=1792, out_features=8064, bias=False)
      (up_proj): Linear(in_features=1792, out_features=8064, bias=False)
      (down_proj): Linear(in_features=8064, out_features=1792, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((1792,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((1792,), eps=1e-05)
  )
)

### hidden 층 연산
- 정답을 찾기 위한 연산 방법
  - hidden_size * hidden_size 를 통해 정답을 찾는 연산을 한다
- LoRA
  - 위 처럼 연산 시 너무 큰 연산(1792*1792)을 하기 때문에 연산 속도를 빠르게 하기 위해 hidden층의 특정 값에 연산 방식을 변경할 수 있다
  - 사용자 지정 값(8) * hidden_size = 그럼 1792*1792 보다 작게 연산하기 때문에 속도가 빨라진다.
  - 단, 부여가능한 값이 지정되어 있다(q, k, v)

### LoRA 속성
- target_modules : 특정 레이어만 가중치를 부여하는 공간
  - 모델마다 정해져 있다
- r
  - 보편적으로 8 ~ 16 사이의 값 설정
  - r작음 - 조금만 수정, r큼 - 많이 수정
  - 값이 커질수록 배열 크기가 커지기 때문에 더 정교하게(많은) 연산을 해 loss를 더 줄일 수 있다.
  - 값이 크면 메모리 사용량이 증가하고 과적합이 생길 가능성도 있다
- lora_alpha
  - 보편적으로 16 ~ 32 설정 또는 r * 2 설정
  - 값이 커지면 가중치의 값을 크게 주게 되어 학습 속도는 빠를 수 있지만 loss는 증가할 수 있다.
  - r과 관계성을 가지고 있으면 두 숫자를 잘 조정하여 사용한다.
- lora_dropout
  - 총 연산에서 연산하지 않을 수( 0.1 => 10% 연산 제외 )
  - overfitting(과적합) 방지용( 보편적인 값 : 0.05~0.2 )
  - 값이 커지면 성능은 저하된다.(과대적합을 피할 수 있다)
  - 값이 작으면 성능은 좋아지나 과대적합 위험이 있다
  - 처리 속도는 거의 무관하다고 보면 된다
- task_type : 모델 학습 목표(task type).
  - "CAUSAL_LM" → 일반적인 언어 생성 모델 (GPT 계열)
  - "SEQ_2_SEQ_LM" → Encoder-Decoder 모델 (BART, T5 등)
  - "TOKEN_CLASSIFICATION" → 토큰 단위 분류
  - "SEQ_CLS" → 문장/문서 단위 분류
---
### LoRA는 전체 모델을 학습하지 않고 일부 레이어만 효율적으로 학습하도록 하는 기법
- r, lora_alpha → 얼마나 강하게 학습할지 결정
- target_modules → 적용할 레이어 선택
- lora_dropout → overfitting 방지
- task_type → 모델 학습 목표 지정

In [3]:
from peft import LoraConfig, get_peft_model

In [4]:
model.model.layers

ModuleList(
  (0-31): 32 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=1792, out_features=3072, bias=False)
      (k_proj): Linear(in_features=1792, out_features=1024, bias=False)
      (v_proj): Linear(in_features=1792, out_features=1024, bias=False)
      (o_proj): Linear(in_features=3072, out_features=1792, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=1792, out_features=8064, bias=False)
      (up_proj): Linear(in_features=1792, out_features=8064, bias=False)
      (down_proj): Linear(in_features=8064, out_features=1792, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((1792,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((1792,), eps=1e-05)
  )
)

In [5]:
lora_config = LoraConfig(
    target_modules=['q_proj','k_proj','v_proj'], # 로라 적용 레이어(위치)
    r = 8, # 연산할 벡터 길이(1792 대신 8로 설정)
    lora_alpha = 16, # 로라 가중치
    lora_dropout = 0.1, # 과적합 방지용, 10% 연산하지 않음
    task_type = 'CAUSAL_LM'
    )


In [6]:
for name, module in model.named_modules():
    print(name)
    break

In [7]:
model_lora = get_peft_model(model, lora_config)

In [8]:
for name, module in model_lora.named_modules():
    print(name)
    #break


base_model
base_model.model
base_model.model.model
base_model.model.model.embed_tokens
base_model.model.model.layers
base_model.model.model.layers.0
base_model.model.model.layers.0.self_attn
base_model.model.model.layers.0.self_attn.q_proj
base_model.model.model.layers.0.self_attn.q_proj.base_layer
base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.model.layers.0.self_attn.q_proj.lora_A
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.model.layers.0.self_attn.q_proj.lora_B
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.model.layers.0.self_attn.q_proj.lora_embedding_A
base_model.model.model.layers.0.self_attn.q_proj.lora_embedding_B
base_model.model.model.layers.0.self_attn.q_proj.lora_magnitude_vector
base_model.model.model.layers.0.self_attn.k_proj
base_model.model.model.layers.0.self_attn.k_proj.base_layer
base_model.mode

### 추론 및 파인 튜닝

In [9]:
prompt = "발주 수량을 어떻게 추천해?"
input = tokenizer(prompt, return_tensors="pt").to("cuda")

In [10]:
outputs = model_lora.generate(
    **input,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.4,
    pad_token_id=tokenizer.eos_token_id
)

In [11]:
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer

'발주 수량을 어떻게 추천해? - 발주 수량 추천\n발주 수량을 어떻게 추천해? - 발주 수량 추천 1. 발주 수량 추천의 중요성 발주 수량 추천은 프로젝트의 성공과 효율성에 큰 영향을 미치는 중요한 요소입니다. 적절한 발주 수량을 추천하면 프로젝트의 비용을 절감하고, 자원의 효율적인 활용을 도모할 수 있습니다. 또한, 발주 수량 추천은 고객과의'

# 미세조정할 학습 데이터 생성
train_data = [
    {"text": "발주 수량을 어떻게 추천해?\n최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다."},
    {"text": "편의점 재고 부족 원인이 뭐야?\n최근 4주 판매량 증가와 발주 리드타임 지연이 동시에 발생했기 때문입니다."},
]

In [12]:
# 미세조정 할 학습 데이터 생성
train_data = [
    {"text": "발주 수량을 어떻게 추천해?\n최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다."},
    {"text": "편의점 재고 부족 원인이 뭐야?\n최근 4주 판매량 증가와 발주 리드타임 지연이 동시에 발생했기 때문입니다."},
]

In [13]:
def tokenize_function(example):
    return tokenizer(example['text'])

In [14]:
from datasets import Dataset

In [15]:
dataset = Dataset.from_list(train_data)
dataset = dataset.map (tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [16]:
dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [17]:
tokenizer.pad_token = tokenizer.eos_token

In [18]:
from transformers import DataCollatorForLanguageModeling

In [19]:
#토큰 길이 및 labels 자동 생성 가능
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
    )

In [20]:
from transformers import TrainingArguments, Trainer

In [21]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 100,
    per_device_train_batch_size = 2,
    report_to = "none",
    fp16 = True,
)

In [22]:
trainer = Trainer(
    model = model_lora,
    args = training_args,
    train_dataset = dataset,
    data_collator = data_collator
)

In [23]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=100, training_loss=1.2786868286132813, metrics={'train_runtime': 37.8174, 'train_samples_per_second': 5.289, 'train_steps_per_second': 2.644, 'total_flos': 87040116633600.0, 'train_loss': 1.2786868286132813, 'epoch': 100.0})

In [24]:
prompt = "발주 수량을 어떻게 추천해?"
input= tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model_lora.generate(
    **input,
    max_new_tokens = 100,
    do_sample = True,
    top_p = 0.9,
    temperature = 0.4,
    pad_token_id = tokenizer.eos_token_id
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer

'발주 수량을 어떻게 추천해? 추천 알고리즘을 만들어야겠어!\n최근 들어 매출이 급증했는데, 어떤 상품을 함께 판매하면 좋을지 추천해줘. 최근 들어 매출이 급증했는데, 어떤 상품을 함께 판매하면 좋을지 추천해줘. 최근 들어 매출이 급증했는데, 어떤 상품을 함께 판매하면 좋을지 추천해줘. 최근 들어 매출이 급증했는데, 어떤 상품을 함께 판매하면 좋'

# 서빙(Serving)
- 학습된 모델을 불러와서 실제로 추론에 활용하는 과정
- 즉, 모델을 단순히 학습만 시키는 게 아니라
  - 사용자 입력(프롬프트, 데이터)을 받아서
  - 모델이 결과(답변, 분류 결과 등)을 내도록 하는 단계
### 1.로컬 서빙(오프라인 서빙)
- 지금 처럼 colab/python등 코드에서 실행
- 개인 환경에서 직접 실행하는 것
### 2.API 서빙(온라인 서빙)
- 모델을 웹 서버에 올려두고 REST API 같은 방식으로 외부에서 요청을 받아 추론 실행하는 방법
- 요청 시 모델이 답변 반환하는 방식

In [25]:
!ls

results  sample_data


In [26]:
merged_model = model_lora.merge_and_unload()

In [27]:
# safe_serialization : 확장자 지정
merged_model.save_pretrained("./lora_model", safe_serialization=True)
tokenizer.save_pretrained("./lora_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./lora_model/tokenizer_config.json',
 './lora_model/chat_template.jinja',
 './lora_model/tokenizer.json')

In [28]:
!ls

lora_model  results  sample_data


### 모델 로드 및 추론

In [29]:
#모델 로드
model_id = "./lora_model"
load_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
load_token = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [30]:
prompt = "발주 수량을 어떻게 추천해?"
def generate_answer(prompt):
  input= load_token(prompt, return_tensors="pt").to("cuda")
  outputs = load_model.generate(
      **input,
      max_new_tokens = 100,
      do_sample = True,
      top_p = 0.9,
      temperature = 0.4,
      pad_token_id = tokenizer.eos_token_id
  )

  answer = load_token.decode(outputs[0], skip_special_tokens=True)
  return answer

In [31]:
prompt = "발주 수량을 어떻게 추천해?"
generate_answer(prompt)

'발주 수량을 어떻게 추천해? 추천 알고리즘을 만들어야겠어!\n안녕하세요! 오늘은 추천 알고리즘에 대해 이야기해보려고 합니다. 최근에 제가 판매하는 상품의 판매량이 급증하면서, 어떤 상품을 추천해줘야 할지 고민이 많아졌어요. 그래서 이번에는 추천 알고리즘을 만들어보려고 합니다. 함께 알아보시죠! 추천 알고리즘의 개념 추천 알고리즘은 사용자의 과거 구매 이력, 상품의 특'

In [32]:
next(model_lora.parameters()).dtype

torch.bfloat16

### QLoRA(Quantized LoRA)
- LoRA + 4bit 양자화 결합한 기술(메모리 8배 절약)
- 큰 언어 모델(LLM)을 메모리를 효율적으로 미세조정 할 때 사용
- 양자화
  - 모델 파라미터를 32bit -> 4bit로 압축
  - 메모리 사용량을 획기적으로 줄임
  - 계산 정확도는 약간 손실 가능성 존재하지만, LoRA로 학습하면 크게 문제는 없다
- 결론
  - QLoRa는 결과적으로 모델을 양자화(크기를 줄인다)를 진행한 모델이 된다.
  - 이는 추론만 하는 경우 사용할 수 있으며, 이 값을 학습하고자 하는 경우 LoRA와 함께 사용하게 된다

In [37]:
import torch
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # 4bit 형식으로 모델 압축
    bnb_4bit_use_double_quant=True, #위의 압축 모델크기에서 한번 더 압축(메모리 효율)
    bnb_4bit_quant_type="nf4", #양자화 설정 시 보편적으로 nf4 사용, 추론에 좋음
    bnb_4bit_compute_dtype=torch.float16 #파라미터 16 연산
)

In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "kakaocorp/kanana-nano-2.1b-base"
tokenizer = AutoTokenizer.from_pretrained( model_id )
model = AutoModelForCausalLM.from_pretrained(
    model_id ,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
     )

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [39]:
#QLora 설정 시 거의 필수로 지정
#메모리 절약효과, 체크포인트 중간 생략하여 연산
#연산속도는 조금 느릴 수 있음
model.gradient_checkpointing_enable()

In [40]:
!ls

lora_model  results  sample_data


In [41]:
model.save_pretrained("./q_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [42]:
from peft import prepare_model_for_kbit_training, LoraConfig
lora_config = LoraConfig(
    target_modules=['q_proj','k_proj','v_proj'] , #로라 적용 레이어(위치)
    r = 8, #연산할 백터 길이(1792 대신 8로 설정 )
    lora_alpha = 16, #가중치 조정
    lora_dropout=0.1, # 과적합 방지용, 10% 연산하지 않음
    task_type = 'CAUSAL_LM'
)
# qLoRa 만들고자 할때 미세조정을 조금 더 잘할 수 있게 설정
q_model = prepare_model_for_kbit_training(model)

In [43]:
q_model = get_peft_model(q_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [44]:
train_data = [
{"text": "### 질문: 발주 수량을 어떻게 추천해?\n### 답변: 최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다."},
{"text": "### 질문: 편의점 재고 부족 원인이 뭐야?\n### 답변: 최근 4주 판매량 증가와 발주 리드타임 지연이 동시에 발생했기 때문입니다."},
]

In [45]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

#토큰화 함수 생성
def tokenize_function(example):
    tokens = tokenizer( example["text"])
    return tokens

dataset = Dataset.from_list(train_data)
dataset = dataset.map( tokenize_function )

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
#pad_token 설정
tokenizer.pad_token = tokenizer.eos_token

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

### TrainingArguments
- 학습 과정에서 사용할 매개변수
- output_dir : 코랩이 관리하는 위치 /content/폴더 저장
  - 저장된 내용을 토대로 학습 모델을 가져와 사용할 수 있음
- num_train_epochs : 총 학습 반복 횟수
  - 1000개의 데이터가 있다고 가정했을 경우
    - 1이면 1000개의 데이터 한번 학습
    - 2이면 1000개의 데이터 2번 학습
  - 즉, epoch를 적정하게 맞추면 과소, 과대 적합을 피할 수 있고 학습률이 좋아진다. 단, 시간이 걸림
- train_batch_size : 한번에 학습할 데이터 양
  - 2인경우 2개의 데이터씩 묶어서 학습( 지금 학습 데이터는 총 2개가 존재(배열 기준))
    - 배치 크기가 작은 경우
      - 메모리 적게 사용
      - 학습 속도 느림
      - 일반화 성능 좋음
    - 배치 크기가 큰 경우
      - 학습 속도 빠름
      - 메모리 사용량 증가
      - 일반화 성능 떨어짐
- learning_rate : 모델이 예측값과 실제값 사이의 오차를 보고 파라미터 값을 얼마나 크게 변경할지 결정하는 계수
  - 작으면(1e-6) 안정적이고, loss를 천천히 감소. 학습 느림
  - 적당(1e-5, 5e-5) 안정적이고, 빠른 수렴, 적당히 학습 빠름
  - 크면(1e-3 이상) 빨리 수렴, 학습 불안정
  - 5e-5 : 0.00005 의미( -5가 소수점 자리 )

In [46]:
from transformers import Trainer, TrainingArguments

In [47]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs =50,
    per_device_train_batch_size = 2,
    learning_rate = 5e-5,
    report_to = "none",
)

In [48]:
!ls

lora_model  q_model  results  sample_data


In [49]:
trainer = Trainer(
    model = q_model,
    train_dataset = dataset,
    args = training_args,
    data_collator = data_collator
)

In [50]:
import torch
print(torch.cuda.is_available())  # True여야 정상
print(torch.cuda.get_device_name(0))  # GPU 이름 출력

True
Tesla T4


In [51]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=50, training_loss=0.7043746948242188, metrics={'train_runtime': 23.7292, 'train_samples_per_second': 4.214, 'train_steps_per_second': 2.107, 'total_flos': 49099552972800.0, 'train_loss': 0.7043746948242188, 'epoch': 50.0})

In [52]:
def generate_answer( prompt ):
  input= tokenizer(prompt, return_tensors="pt").to("cuda")
  outputs = q_model.generate(
      **input,
      max_new_tokens = 100,
      do_sample = True,
      top_p = 0.9,
      temperature = 0.4,
      pad_token_id = tokenizer.eos_token_id
  )
  answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return answer
#발주 수량을 어떻게 추천해?\n최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다

In [53]:
answer = generate_answer("발주 수량을 어떻게 추천해?")
answer

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


'발주 수량을 어떻게 추천해? 추천지 발기기와 발발발기기가 수 있수이 지�기발전발발발발발기발기 발기기가 편서 발리발발지발어발발발발발발발발발가 발발발발발발기고 발치상발발발상발기기고 발�발발서를 발상고 발고기지발상이 발발발발발발발발발치기로 발'

### 저장 모델 크기 확인

In [54]:
model_id = "kakaocorp/kanana-nano-2.1b-base"
mod = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [55]:
!ls

lora_model  q_model  results  sample_data


In [56]:
mod.save_pretrained("./model")

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3336: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!ls

In [ ]:
import os
mod_path = "./model"
q_mod_path = "./q_model"

mod_list = list(os.walk(mod_path))
q_mod_list = list(os.walk(q_mod_path))

print(mode_list)
print(q_mod_list)
#[ 현재 경로, [하위폴더], [실제 파일명] ]

In [36]:
def get_model_size(path, fileList):
  total_size = 0
  for f in fileList:
    fp = os.path.join(path, f)
    total_size += os.path.getsize(fp)
  #total_size : byte 크기 1byte * 1024 => KB, 1KB * 1024 => 1MB
  return total_size

SyntaxError: invalid syntax (3030966602.py, line 4)

In [ ]:
print("model size : ", get_model_size(mod_path, mod_list[0],[2]))
print("q_model size : ", get_model_size(q_model_path, q_model_list[0],[2]))